<a href="https://colab.research.google.com/github/ben854719/Patient-Electronic-Health-Record-System/blob/main/Agentic_AI_Patient_Record.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade langchain-google-genai google-generativeai
!pip install --upgrade langchain-google-genai google-generativeai langgraph

INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
  Using cached langchain_google_genai-2.1.8-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_ai_generativelanguage-0.6.18-py3-none-any.whl.metadata (9.8 kB)
INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
  Using cached google_generativeai-0.8.5-

In [4]:
from ast import Try
from IPython import get_ipython
from IPython import display
import os
from langgraph.graph import StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import SystemMessage, HumanMessage
from typing import TypedDict, List
from google.colab import userdata

# Import google colab.
Colab_Secret_key = "Ben85"

# Import API Key to function Gemini.
api_key = userdata.get("Ben85")
if not api_key:
   raise ValueError("Ben85 secret not found. Please set your API key in Colab Secrets with the same Ben85")

# Define the state of schema using TypeDict.
class LogAnalysisState(TypedDict):
  logs: List[str]
  analysis: str
  translated_text: str

# Initialize Gemini model.
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key)

# Create analyze medical log of the patient file.

def analyze_logs(state: LogAnalysisState) -> dict:
  """
  Analyze a list of medical logs of the diagnosis of the patient health condition using Gemini model.

   Args:
      state: The current state of the LangGraph workflow, containing the logs.

  Returns:
      A dictionary containing the analysis results to update the state. the key 'analysis'
      will hold a string with the detail analysis, potentially included identified
      diagnosis of the health of the patient and their description.
   """
  logs = state['logs']

  # Construct a more detailed prompt for the model.
  prompt_text = (
      "Analyze the following the medical logs of the diagnosis of the patient file carefully. Your task is to identify "
      "Any potential underlying conditions, and any gaps or inconsistencies in the diagnostic process."
      "Any Highlight areas that may require further testing or evaluation, and suggest appropriate next steps for treatment or consultation."
      "For each diagnonis, provide a brief description and indicates either in English or French."
      "which log entries are related.\n\n"
      "Medical Logs:\n" + "\n".join(logs) + "\n\n"
      "Please provide your analysis in a clear and concise manner."
    )

# Invoke the Gemini Model with the prompt wrapped in a human message.
  try:
    response = gemini_model.invoke([HumanMessage(content=prompt_text)])
    analysis_result = response.content
  except Exception as e:
    analysis_result = f"Error during analysis:  {e}"
    display.display(analysis_result)

    # Return a dictionary with the analysis content to update the state.
  return {"analysis": analysis_result}

# Translation medical log of the patient file.
def translate_text(state: LogAnalysisState) -> dict:
   """
   Translate the text to the target language using Gemini.
  _Parameters:
  text(str): The text to be translated from English to French
  target_language(str): The target Language to translate from English To French
  _Returns:
  dict: the translated text
  """
   text_to_translate = state['analysis']
   target_language = "French"

   prompt = f"Translate the following text From English to {target_language}: {text_to_translate}"
   response = gemini_model.invoke([HumanMessage(content=prompt)])
   return {"translated_text": response.content}

# Create LangGraph workflow.
workflow = StateGraph(state_schema=LogAnalysisState)
workflow.add_node("log_analysis", analyze_logs)
workflow.add_node("translation", translate_text)
workflow.add_edge("log_analysis", "translation")
workflow.set_entry_point("log_analysis")

# Compile the workflow.
app = workflow.compile()

# Create a medical log.
medical_log = [
    "The profile of the patient",
    "The age of the patient is 43",
    "The gender of the patient",
    "The medical history of the patient has diabetes",
    "The diagnosis health condition",
    "The social class of the patient is middle class",
    "The name of the patient is Mary Smith Wong",
    "The name of the parents of Mary are Tony Wong and Rose-Mary Smith",
    "Mary was born in April 8, 1983 in the city of Toronto",
    "The patient is a mix race",
    "The name of the patience husband is Eric Taylor"
    "The patient is a overwheight",
    "The patient is a heavy smoker",
    "The patience is marry to her husband",
    "The patience does not do exercise",
    "The patience has 3 childrens. She has 2 boys and 1 girl",
    "The name of her sons are John Taylor and Ben Taylor",
    "The name of her daughter is Erica Taylor",
    "The province lives in Ontario",
]

# Run the workflow>
display.display("Running log analysis workflow...")
result = app.invoke({"logs": medical_log})
display.display("\nAnalysis Result:")
display.display(result['analysis'])
display.display(f"Translated text (French): {result['translated_text']}")

# Print original and translated text
print(f"Original text: {result['analysis']}")
print(f"Translated text (French): {result['translated_text']}")

'Running log analysis workflow...'

'\nAnalysis Result:'

'This medical log provides a patient profile rather than a detailed diagnostic workup or presenting complaints. Therefore, the analysis will focus on identifying significant risk factors, potential underlying conditions implied by these factors, and critical gaps in the information provided.\n\n---\n\n### **Analyse des Dossiers Médicaux de Mary Smith Wong**\n*(Analysis of Mary Smith Wong\'s Medical Logs)*\n\n**Patient Profile Overview:**\nMary Smith Wong, a 43-year-old (with an age/DOB discrepancy) mixed-race female residing in Ontario, is married with three children. She has a history of diabetes, is overweight, and is a heavy smoker. She also reports no regular exercise. Her social class is middle class.\n\n---\n\n### **Potential Underlying Conditions & Key Risk Factors**\n*(Conditions Sous-jacentes Potentielles et Facteurs de Risque Clés)*\n\nBased on the provided information, Mary presents with a high-risk profile for several serious health conditions.\n\n1.  **Diabète (Diabetes)**

'Translated text (French): Voici la traduction du texte de l\'anglais vers le français :\n\nCe dossier médical fournit un profil de patiente plutôt qu\'un bilan diagnostique détaillé ou des motifs de consultation. Par conséquent, l\'analyse se concentrera sur l\'identification des facteurs de risque significatifs, des conditions sous-jacentes potentielles impliquées par ces facteurs, et des lacunes critiques dans les informations fournies.\n\n---\n\n### **Analyse des Dossiers Médicaux de Mary Smith Wong**\n*(Analysis of Mary Smith Wong\'s Medical Logs)*\n\n**Aperçu du Profil de la Patiente :**\nMary Smith Wong, une femme métisse âgée de 43 ans (avec une divergence âge/date de naissance) résidant en Ontario, est mariée et a trois enfants. Elle a des antécédents de diabète, est en surpoids et est une grosse fumeuse. Elle déclare également ne pas faire d\'exercice régulièrement. Sa classe sociale est la classe moyenne.\n\n---\n\n### **Conditions Sous-jacentes Potentielles et Facteurs de R

Original text: This medical log provides a patient profile rather than a detailed diagnostic workup or presenting complaints. Therefore, the analysis will focus on identifying significant risk factors, potential underlying conditions implied by these factors, and critical gaps in the information provided.

---

### **Analyse des Dossiers Médicaux de Mary Smith Wong**
*(Analysis of Mary Smith Wong's Medical Logs)*

**Patient Profile Overview:**
Mary Smith Wong, a 43-year-old (with an age/DOB discrepancy) mixed-race female residing in Ontario, is married with three children. She has a history of diabetes, is overweight, and is a heavy smoker. She also reports no regular exercise. Her social class is middle class.

---

### **Potential Underlying Conditions & Key Risk Factors**
*(Conditions Sous-jacentes Potentielles et Facteurs de Risque Clés)*

Based on the provided information, Mary presents with a high-risk profile for several serious health conditions.

1.  **Diabète (Diabetes)**
   